In [1]:
# Author: Gergely Zahoranszky-Kohalmi, PhD
#
# Email: gergely.zahoranszky-kohalmi@nih.gov
#
# Organization: National Center for Advancing Translational Sciences
#


In [2]:
# Env: syngps_rev
import syngps
from syngps import SynthGraph, AicpFunctions, graph_utils as gu
import json
import networkx as nx
import pandas as pd

import time

import os

In [3]:
DIR_IN = '../data/output/synthesis_graphs_inv/'

FNAME_IN_ALL = []

FNAME_OUT = '../data/output/route_finding_with_inventory_timing_stats_rxsmiles.tsv'

#sg_assembly_durations = []          # in seconds
#synth_route_finding_durations = []  # in seconds
#tms = []

SG_ASSEMBLY_DURATIONS = []          # in seconds
SYNTH_ROUTE_FINDING_DURATIONS = []  # in seconds
TMs = []




In [4]:
# Functions




def identify_first_synthesis_route (fname_json):
    
    first = True

    time_start = time.time()

    routes_json = None




    print (f'[*] Input file: {fname_json}')

    try:

        with open(fname_json) as f:
            routes_json = json.load(f)
            #print (f'{routes_json}')
            #print(json.dumps(routes_json, indent=4))
    
    except:
        print (f'[W] No synthesis graph found in JSON response: {fname_json}')

    #    return (None)

    if routes_json != None:
        G_sg = syngps.utils.json_to_graph(routes_json)
        #print (G_sg)

        TMID = routes_json['search_params']['target_molecule_inchikey']

        time_sg = routes_json['time_info']['synth_graph_time']
        
        

        # Generatig the artifact-free synthesis graph
        G_sg_af = gu.remove_incompatible_reactions (G_sg.copy(), target_molecule_node_id = TMID)

        # Generating combination graphs
        # and allowing to find all combination graphs based on the edge-betweenness centrality (EBC) method
        G_combinations = []
        G_combinations = gu.generate_combination_graphs (G_sg_af.copy(), method = "ebc", max_nr = 0)



        # Identifying Viable Synthesis Routes (VSRs)
        VSRs = []

        for C in G_combinations:

            R = gu.find_viable_route (C, target_module_node_id = TMID)
            
            if R[1] == 'Viable Route Candidate' or R[1] == 'Viable Synthesis Route':
                time_end = time.time()

                if first:
                    
                    print (f'[*] Duration (s): {time_end - time_start} , SG fetching time: {time_sg}.')
                    
                    SYNTH_ROUTE_FINDING_DURATIONS.append(time_end - time_start)
                    
                    SG_ASSEMBLY_DURATIONS.append (time_sg)
                    
                    TMs.append(TMID)

                    first = False

                #VSRs.append(R[0])
                #print (R[0])
            


    else:


        print (f'[W] No synthesis route found in JSON response: {fname_json}')

    return (None)


#####






In [5]:
search_obj = os.scandir(DIR_IN)

for dir_item in search_obj:
    if dir_item.is_file():
        
        FNAME_IN_ALL.append(DIR_IN + dir_item.name)

print(FNAME_IN_ALL)

['../data/output/synthesis_graphs_inv/DUCJUKZQSRPECI-UHFFFAOYSA-N_response.json', '../data/output/synthesis_graphs_inv/UZCLUOWLQYOFSZ-UHFFFAOYSA-N_response.json', '../data/output/synthesis_graphs_inv/WTCXKDSZUHYCLH-UHFFFAOYSA-N_response.json', '../data/output/synthesis_graphs_inv/FMXLKQYOBUDPPG-UHFFFAOYSA-N_response.json', '../data/output/synthesis_graphs_inv/FOROUCWBDGYYOS-NSHDSACASA-N_response.json', '../data/output/synthesis_graphs_inv/IWJOUZQQKDORMF-UHFFFAOYSA-N_response.json', '../data/output/synthesis_graphs_inv/MGGGSJRWNQFCLE-UHFFFAOYSA-N_response.json', '../data/output/synthesis_graphs_inv/IXZILUICVMZFAS-UHFFFAOYSA-N_response.json', '../data/output/synthesis_graphs_inv/GZYNUOMXGGSXMI-UHFFFAOYSA-N_response.json', '../data/output/synthesis_graphs_inv/WEEVXEKXCIBZMB-UHFFFAOYSA-N_response.json', '../data/output/synthesis_graphs_inv/FWLFRXVPNVBXGR-CQSZACIVSA-N_response.json', '../data/output/synthesis_graphs_inv/MRPFJQLRQGTKNI-UHFFFAOYSA-N_response.json', '../data/output/synthesis_g

In [6]:
for fname in FNAME_IN_ALL:

    
    identify_first_synthesis_route (fname)



[*] Input file: ../data/output/synthesis_graphs_inv/DUCJUKZQSRPECI-UHFFFAOYSA-N_response.json
[*] Input file: ../data/output/synthesis_graphs_inv/UZCLUOWLQYOFSZ-UHFFFAOYSA-N_response.json
[*] Duration (s): 0.00145721435546875 , SG fetching time: 0.010889291763305664.
[*] Input file: ../data/output/synthesis_graphs_inv/WTCXKDSZUHYCLH-UHFFFAOYSA-N_response.json
[*] Input file: ../data/output/synthesis_graphs_inv/FMXLKQYOBUDPPG-UHFFFAOYSA-N_response.json
[*] Input file: ../data/output/synthesis_graphs_inv/FOROUCWBDGYYOS-NSHDSACASA-N_response.json
[*] Input file: ../data/output/synthesis_graphs_inv/IWJOUZQQKDORMF-UHFFFAOYSA-N_response.json
[*] Input file: ../data/output/synthesis_graphs_inv/MGGGSJRWNQFCLE-UHFFFAOYSA-N_response.json
[*] Input file: ../data/output/synthesis_graphs_inv/IXZILUICVMZFAS-UHFFFAOYSA-N_response.json
[*] Duration (s): 0.09691691398620605 , SG fetching time: 3.165074110031128.
[*] Input file: ../data/output/synthesis_graphs_inv/GZYNUOMXGGSXMI-UHFFFAOYSA-N_response.js

In [7]:
df = pd.DataFrame({
    'tm': TMs,
    'route_find_time_sec': SYNTH_ROUTE_FINDING_DURATIONS,
    'synth_graph_assembly_time_sec': SG_ASSEMBLY_DURATIONS
})

print (df)


df.to_csv (FNAME_OUT, sep = '\t', index = False)

print ('[Done.]')

                             tm  route_find_time_sec  \
0   UZCLUOWLQYOFSZ-UHFFFAOYSA-N             0.001457   
1   IXZILUICVMZFAS-UHFFFAOYSA-N             0.096917   
2   GZYNUOMXGGSXMI-UHFFFAOYSA-N             0.006690   
3   WEEVXEKXCIBZMB-UHFFFAOYSA-N             0.005135   
4   VLYBAPCPKDLKOF-UHFFFAOYSA-N             0.000691   
5   KJDKPNUDCPAPKM-UHFFFAOYSA-N             0.035454   
6   HDQNXIOVYVAKBX-UHFFFAOYSA-N             0.000629   
7   VTJXLVGZNOINAD-UHFFFAOYSA-N             0.001530   
8   BVKGNQRDVFGNIW-UHFFFAOYSA-N             0.008715   
9   IHXNMNXXLBANRT-UHFFFAOYSA-N             0.001582   
10  DJRUWESABPZMRB-UHFFFAOYSA-N             0.005839   
11  PBLNHHSDYFYZNC-UHFFFAOYSA-N             0.032476   
12  FNMOSYNODGOFRC-AWEZNQCLSA-N             0.000571   
13  OPYBJNYVSLHSQI-UHFFFAOYSA-N             0.005511   
14  MGPKTKXUWSVAMY-UHFFFAOYSA-N             0.009458   
15  IQNMCFJVBQJVNL-UHFFFAOYSA-N             0.000895   
16  QHQSWYIJDFMBNE-UHFFFAOYSA-N             0.00

In [8]:
# References:
#
# Ref: https://stackoverflow.com/questions/20199126/reading-json-from-a-file
# Ref: https://stackoverflow.com/questions/12943819/how-to-prettyprint-a-json-file
# Ref: Google AI Overview, 04/30/2026, by Google Gemini AI built-in Chrome
# Ref: https://www.geeksforgeeks.org/python/python-time-module/
#


